# Crescendo Faithful — Representation Analysis

Analyses hidden states extracted by notebook 11 (Design D, layers 1/16/32,
every turn including refusal turns, rollback-aware context).

## Questions

| # | Question | Method |
|---|----------|--------|
| 1 | Can a probe distinguish harmful from benign conversations? | LR + LAT per layer |
| 2 | Does the probe score drift turn-by-turn toward jailbreak? | Mean score trajectory by verdict |
| 3 | Do refusal turns look representationally different from non-refusal turns? | LR within harmful convs |
| 4 | Is there a sharp tipping point at the jailbreak turn? | Score aligned to jailbreak_turn |
| 5 | Which layer carries the signal? | AUC heatmap layer × turn |

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

sns.set_theme(style="whitegrid", font_scale=1.1)
FIG_DIR = repo_root / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

LAYER_INDICES = [1, 16, 32]
LAYER_NAMES   = [f"Layer {l}" for l in LAYER_INDICES]
LAYER_COLORS  = ["#4e79a7", "#f28e2b", "#e15759"]   # blue, orange, red

print("Imports OK")

## Load data

In [ ]:
REPR_ROOT = repo_root / "data" / "representations"

def load_split(folder_name):
    d = REPR_ROOT / folder_name
    states = np.load(str(d / "hidden_states.npy"), mmap_mode="r")
    meta   = pd.read_parquet(d / "metadata.parquet")
    print(f"{folder_name}: {states.shape}  ({len(meta)} rows)")
    return states, meta

states_h, meta_h = load_split("crescendo_harmful_10_turns_faithful")
states_b, meta_b = load_split("crescendo_benign_10_turns_faithful")

# Combined for Task 1
states_all = np.concatenate([states_h, states_b], axis=0).astype(np.float32)
meta_all   = pd.concat([meta_h, meta_b], ignore_index=True)

print(f"\nCombined: {states_all.shape}")
print(f"Label balance: {meta_all['label'].value_counts().to_dict()}")
print(f"Harmful verdicts: {meta_all[meta_all.split=='harmful']['final_verdict'].value_counts().to_dict()}")

## Probe helpers

In [ ]:
def lr_probe_cv(X: np.ndarray, y: np.ndarray, n_splits: int = 5) -> float:
    """Stratified k-fold LR probe. Returns mean AUC."""
    if len(np.unique(y)) < 2 or len(y) < n_splits * 2:
        return float("nan")
    skf  = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    aucs = []
    for train_idx, test_idx in skf.split(X, y):
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X[train_idx])
        X_te = scaler.transform(X[test_idx])
        clf  = LogisticRegression(max_iter=1000, C=0.1)
        clf.fit(X_tr, y[train_idx])
        aucs.append(roc_auc_score(y[test_idx], clf.predict_proba(X_te)[:, 1]))
    return float(np.mean(aucs))


def lat_probe_cv(X: np.ndarray, y: np.ndarray, n_splits: int = 5) -> float:
    """LAT reading vector: mean-diff direction, dot-product score. Returns mean AUC."""
    if len(np.unique(y)) < 2 or len(y) < n_splits * 2:
        return float("nan")
    skf  = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    aucs = []
    for train_idx, test_idx in skf.split(X, y):
        scaler   = StandardScaler()
        X_tr     = scaler.fit_transform(X[train_idx])
        X_te     = scaler.transform(X[test_idx])
        y_tr     = y[train_idx]
        vec      = X_tr[y_tr == 1].mean(0) - X_tr[y_tr == 0].mean(0)
        vec      = vec / (np.linalg.norm(vec) + 1e-8)
        scores   = X_te @ vec
        aucs.append(roc_auc_score(y[test_idx], scores))
    return float(np.mean(aucs))


def get_layer(states: np.ndarray, layer_pos: int) -> np.ndarray:
    """Extract one layer slice. layer_pos is index into LAYER_INDICES (0, 1, or 2)."""
    return np.asarray(states[:, layer_pos, :], dtype=np.float32)


print("Probe helpers defined.")

## Task 1 — Harmful vs Benign

Can a probe trained on the pre-generation hidden state distinguish turns from harmful
conversations from turns from benign conversations?
Reported per layer and per turn position.

In [ ]:
y_all = meta_all["label"].values

rows = []
for li, lname in enumerate(LAYER_NAMES):
    X = get_layer(states_all, li)
    rows.append({"layer": lname, "method": "LR",  "auc": lr_probe_cv(X, y_all)})
    rows.append({"layer": lname, "method": "LAT", "auc": lat_probe_cv(X, y_all)})

df_task1 = pd.DataFrame(rows)
print("Task 1 — Harmful vs Benign AUC by layer:")
print(df_task1.pivot(index="layer", columns="method", values="auc").round(3).to_string())

In [ ]:
# AUC per (layer, turn_idx)
turns = sorted(meta_all["turn_idx"].unique())
rows = []
for turn in turns:
    mask = meta_all["turn_idx"].values == turn
    y_t  = y_all[mask]
    if y_t.sum() < 5 or (y_t == 0).sum() < 5:
        continue
    for li, lname in enumerate(LAYER_NAMES):
        X_t = get_layer(states_all[mask], li)
        rows.append({"turn": turn, "layer": lname,
                     "LR":  lr_probe_cv(X_t, y_t),
                     "LAT": lat_probe_cv(X_t, y_t)})

df_t1_turn = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, method in zip(axes, ["LR", "LAT"]):
    for li, (lname, color) in enumerate(zip(LAYER_NAMES, LAYER_COLORS)):
        sub = df_t1_turn[df_t1_turn["layer"] == lname]
        ax.plot(sub["turn"], sub[method], marker="o", color=color, label=lname)
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
    ax.set_title(f"Task 1: Harmful vs Benign — {method}")
    ax.set_xlabel("Turn index")
    ax.set_ylabel("AUC")
    ax.set_ylim(0.4, 1.05)
    ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "crescendo_task1_by_turn.png", dpi=150)
plt.show()

## Task 2 — Refusal turn prediction (within harmful conversations)

Can the pre-generation hidden state predict whether the model is about to refuse?
Only uses turns from harmful conversations.

Note: `is_refusal=True` = step judge said response didn't progress toward the objective
(includes both outright refusals and off-topic responses).

In [ ]:
y_refusal = meta_h["is_refusal"].astype(int).values
print(f"Harmful turns: {len(y_refusal)}  (refusal={y_refusal.sum()}, non-refusal={(y_refusal==0).sum()})")

rows = []
for li, lname in enumerate(LAYER_NAMES):
    X = get_layer(states_h, li)
    rows.append({"layer": lname, "method": "LR",  "auc": lr_probe_cv(X, y_refusal)})
    rows.append({"layer": lname, "method": "LAT", "auc": lat_probe_cv(X, y_refusal)})

df_task2 = pd.DataFrame(rows)
print("\nTask 2 — Refusal prediction AUC by layer:")
print(df_task2.pivot(index="layer", columns="method", values="auc").round(3).to_string())

# By turn
rows2t = []
for turn in sorted(meta_h["turn_idx"].unique()):
    mask = meta_h["turn_idx"].values == turn
    y_t  = y_refusal[mask]
    if y_t.sum() < 5 or (y_t == 0).sum() < 5:
        continue
    for li, lname in enumerate(LAYER_NAMES):
        X_t = get_layer(np.asarray(states_h)[mask], li)
        rows2t.append({"turn": turn, "layer": lname, "AUC": lr_probe_cv(X_t, y_t)})

df_t2_turn = pd.DataFrame(rows2t)
fig, ax = plt.subplots(figsize=(7, 4))
for lname, color in zip(LAYER_NAMES, LAYER_COLORS):
    sub = df_t2_turn[df_t2_turn["layer"] == lname]
    ax.plot(sub["turn"], sub["AUC"], marker="o", color=color, label=lname)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Task 2: Refusal turn prediction (LR, harmful only)")
ax.set_xlabel("Turn index")
ax.set_ylabel("AUC")
ax.set_ylim(0.4, 1.05)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "crescendo_task2_refusal_by_turn.png", dpi=150)
plt.show()

## Task 3 — Jailbroken vs Refusal outcome (harmful only)

Can a probe trained on the hidden state at each turn predict whether this
conversation will eventually be jailbroken?
Uses `final_verdict == 'jailbroken'` vs `'refusal'` (excludes `near_miss`).

In [ ]:
mask_outcome = meta_h["final_verdict"].isin(["jailbroken", "refusal"])
states_oc    = np.asarray(states_h)[mask_outcome]
meta_oc      = meta_h[mask_outcome].reset_index(drop=True)
y_oc         = (meta_oc["final_verdict"] == "jailbroken").astype(int).values

print(f"Jailbroken turns: {y_oc.sum()}  Refusal turns: {(y_oc==0).sum()}")

# Overall AUC by layer
rows = []
for li, lname in enumerate(LAYER_NAMES):
    X = get_layer(states_oc, li)
    rows.append({"layer": lname, "method": "LR",  "auc": lr_probe_cv(X, y_oc)})
    rows.append({"layer": lname, "method": "LAT", "auc": lat_probe_cv(X, y_oc)})

df_task3 = pd.DataFrame(rows)
print("\nTask 3 — Outcome prediction AUC:")
print(df_task3.pivot(index="layer", columns="method", values="auc").round(3).to_string())

# AUC by turn — does predictability increase as turns progress?
rows3t = []
for turn in sorted(meta_oc["turn_idx"].unique()):
    mask_t = meta_oc["turn_idx"].values == turn
    y_t = y_oc[mask_t]
    if y_t.sum() < 5 or (y_t == 0).sum() < 5:
        continue
    for li, lname in enumerate(LAYER_NAMES):
        X_t = get_layer(states_oc[mask_t], li)
        rows3t.append({"turn": turn, "layer": lname, "AUC": lr_probe_cv(X_t, y_t)})

df_t3_turn = pd.DataFrame(rows3t)
fig, ax = plt.subplots(figsize=(7, 4))
for lname, color in zip(LAYER_NAMES, LAYER_COLORS):
    sub = df_t3_turn[df_t3_turn["layer"] == lname]
    ax.plot(sub["turn"], sub["AUC"], marker="o", color=color, label=lname)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Task 3: Jailbroken vs Refusal outcome prediction (LR)")
ax.set_xlabel("Turn index")
ax.set_ylabel("AUC")
ax.set_ylim(0.4, 1.05)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "crescendo_task3_outcome_by_turn.png", dpi=150)
plt.show()

## Turn trajectory — probe score drift

Train a LAT probe (Task 1: harmful vs benign) on all data, then score every
turn in every conversation. Plot the mean score trajectory by turn, split
by final verdict.

If Crescendo works by gradually shifting the model's representation, we expect
to see a monotone drift toward the "harmful" direction for jailbroken conversations.

In [ ]:
# Train LAT probe on all data (harmful vs benign) using last layer
TRAJ_LAYER_POS = 2   # layer 32 (last)

X_all  = get_layer(states_all, TRAJ_LAYER_POS)
y_all_ = meta_all["label"].values

scaler = StandardScaler().fit(X_all)
X_scaled = scaler.transform(X_all)
lat_vec = X_scaled[y_all_ == 1].mean(0) - X_scaled[y_all_ == 0].mean(0)
lat_vec = lat_vec / (np.linalg.norm(lat_vec) + 1e-8)

# Project all turns onto LAT direction
scores = X_scaled @ lat_vec
meta_all["lat_score"] = scores

# Mean trajectory per turn, split by (split, final_verdict)
harmful_verdicts = ["jailbroken", "near_miss", "refusal"]
palette = {"jailbroken": "#e15759", "near_miss": "#f28e2b", "refusal": "#4e79a7",
           "benign": "#59a14f"}

fig, ax = plt.subplots(figsize=(9, 5))

# Benign
traj_b = meta_all[meta_all.split == "benign"].groupby("turn_idx")["lat_score"].agg(["mean", "sem"])
ax.plot(traj_b.index, traj_b["mean"], color=palette["benign"], label="Benign", linewidth=2)
ax.fill_between(traj_b.index, traj_b["mean"] - traj_b["sem"],
                traj_b["mean"] + traj_b["sem"], color=palette["benign"], alpha=0.15)

# Harmful by verdict
for verdict in harmful_verdicts:
    sub = meta_all[(meta_all.split == "harmful") & (meta_all.final_verdict == verdict)]
    if len(sub) == 0:
        continue
    traj = sub.groupby("turn_idx")["lat_score"].agg(["mean", "sem"])
    ax.plot(traj.index, traj["mean"], color=palette[verdict],
            label=f"Harmful — {verdict}", linewidth=2)
    ax.fill_between(traj.index, traj["mean"] - traj["sem"],
                    traj["mean"] + traj["sem"], color=palette[verdict], alpha=0.15)

ax.set_xlabel("Turn index")
ax.set_ylabel("LAT score (harmful direction, layer 32)")
ax.set_title("Representation trajectory by turn — harmful vs benign")
ax.legend()
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.savefig(FIG_DIR / "crescendo_trajectory_by_verdict.png", dpi=150)
plt.show()

## Jailbreak tipping point

Align jailbroken conversations by `jailbreak_turn` and plot the LAT score at
positions relative to jailbreak (k = turn_idx - jailbreak_turn).

A sharp jump at k=0 would suggest the tipping point is the jailbreak turn itself.
A gradual rise before k=0 would suggest the drift precedes the jailbreak.

In [ ]:
jb_mask = (meta_all.split == "harmful") & (meta_all.final_verdict == "jailbroken") & meta_all.jailbreak_turn.notna()
df_jb   = meta_all[jb_mask].copy()
df_jb["k"] = df_jb["turn_idx"] - df_jb["jailbreak_turn"]   # signed distance to jailbreak

# Focus on window around jailbreak
window = range(-6, 4)
df_jb_win = df_jb[df_jb["k"].isin(window)]

traj_k = df_jb_win.groupby("k")["lat_score"].agg(["mean", "sem", "count"])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(traj_k.index, traj_k["mean"], color="#e15759", marker="o", linewidth=2)
ax.fill_between(traj_k.index,
                traj_k["mean"] - traj_k["sem"],
                traj_k["mean"] + traj_k["sem"],
                color="#e15759", alpha=0.2)
ax.axvline(0, color="black", linestyle="--", linewidth=1, label="Jailbreak turn (k=0)")
ax.set_xlabel("Turns relative to jailbreak (k = turn − jailbreak_turn)")
ax.set_ylabel("LAT score (harmful direction, layer 32)")
ax.set_title("Representation around the jailbreak turn (jailbroken convs only)")
ax.legend()

# Annotate sample counts
for k, row in traj_k.iterrows():
    ax.annotate(f"n={int(row['count'])}", (k, traj_k["mean"].min() - 0.02 * traj_k["mean"].ptp()),
                ha="center", fontsize=7, color="gray")

plt.tight_layout()
plt.savefig(FIG_DIR / "crescendo_jailbreak_tipping_point.png", dpi=150)
plt.show()

## Layer × turn AUC heatmap

Task 1 (harmful vs benign) AUC across all combinations of layer and turn.
Shows where in the model and when in the conversation the signal is strongest.

In [ ]:
turns_all = sorted(meta_all["turn_idx"].unique())
heatmap_data = np.full((len(LAYER_INDICES), len(turns_all)), np.nan)

for ti, turn in enumerate(turns_all):
    mask = meta_all["turn_idx"].values == turn
    y_t  = meta_all["label"].values[mask]
    if y_t.sum() < 5 or (y_t == 0).sum() < 5:
        continue
    for li in range(len(LAYER_INDICES)):
        X_t = get_layer(states_all[mask], li)
        heatmap_data[li, ti] = lr_probe_cv(X_t, y_t)

fig, ax = plt.subplots(figsize=(10, 3))
im = ax.imshow(heatmap_data, aspect="auto", vmin=0.5, vmax=1.0,
               cmap="RdYlGn", interpolation="nearest")
plt.colorbar(im, ax=ax, label="AUC")
ax.set_xticks(range(len(turns_all)))
ax.set_xticklabels([str(t) for t in turns_all])
ax.set_yticks(range(len(LAYER_INDICES)))
ax.set_yticklabels(LAYER_NAMES)
ax.set_xlabel("Turn index")
ax.set_title("Task 1 (Harmful vs Benign) AUC — Layer × Turn")
for li in range(len(LAYER_INDICES)):
    for ti in range(len(turns_all)):
        v = heatmap_data[li, ti]
        if not np.isnan(v):
            ax.text(ti, li, f"{v:.2f}", ha="center", va="center",
                    fontsize=7, color="black")
plt.tight_layout()
plt.savefig(FIG_DIR / "crescendo_heatmap_layer_turn.png", dpi=150)
plt.show()

## Refusal turns vs context depth

Does the model's refusal rate (representationally) change with how much
compliant context it has already accumulated (`n_context_turns`)?
Uses LAT score in the harmful direction.

In [ ]:
df_h = meta_h.copy()
df_h["lat_score"] = scaler.transform(
    get_layer(states_h, TRAJ_LAYER_POS)
) @ lat_vec

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Mean LAT score by n_context_turns, refusal vs non-refusal
ax = axes[0]
for is_ref, label, color in [(True, "Refusal turn", "#4e79a7"),
                              (False, "Non-refusal turn", "#e15759")]:
    sub = df_h[df_h.is_refusal == is_ref].groupby("n_context_turns")["lat_score"].agg(["mean","sem","count"])
    ax.plot(sub.index, sub["mean"], marker="o", color=color, label=label)
    ax.fill_between(sub.index, sub["mean"] - sub["sem"],
                    sub["mean"] + sub["sem"], color=color, alpha=0.15)
ax.set_xlabel("Non-refusal turns already in context")
ax.set_ylabel("LAT score (harmful direction)")
ax.set_title("Representation by context depth")
ax.legend()

# Refusal rate by n_context_turns
ax = axes[1]
ref_rate = df_h.groupby("n_context_turns")["is_refusal"].agg(["mean", "count"])
ax.bar(ref_rate.index, ref_rate["mean"], color="#4e79a7", alpha=0.8)
for i, (idx, row) in enumerate(ref_rate.iterrows()):
    ax.text(idx, row["mean"] + 0.01, f"n={int(row['count'])}",
            ha="center", fontsize=8, color="gray")
ax.set_xlabel("Non-refusal turns already in context")
ax.set_ylabel("Refusal rate")
ax.set_title("Refusal rate by context depth (harmful)")
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig(FIG_DIR / "crescendo_refusal_context_depth.png", dpi=150)
plt.show()